In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)   # empty slices where a whole row was dropped
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

DATA = '../data'
X_train = np.load(f'{DATA}/train_cadences.npy', mmap_mode='r')   # (18000, 6, 16, 128) uint8
X_test  = np.load(f'{DATA}/test_cadences.npy', mmap_mode='r')    # (10000, 6, 16, 128) uint8
train = pd.read_csv(f'{DATA}/train_meta.csv')
test  = pd.read_csv(f'{DATA}/test_meta.csv')
print(X_train.shape, X_test.shape)
print(train.label.value_counts().to_dict())

(18000, 6, 16, 128) (10000, 6, 16, 128)
{'noise': 4211, 'rfi_persistent': 2892, 'rfi_intermittent': 2662, 'radar_chirp': 1849, 'satellite_pass': 1805, 'pulsar': 1629, 'scintillating_star': 1431, 'frb': 1042, 'eti_candidate': 479}


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report

DATA_DIR = Path("../data")
FEATURE_DIR = Path("../outputs/features")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA_DIR / "train_meta.csv")
test = pd.read_csv(DATA_DIR / "test_meta.csv")

F_e2v2_train = pd.read_pickle(
    FEATURE_DIR / "F_e2v2_train.pkl"
)
F_e2v2_test = pd.read_pickle(
    FEATURE_DIR / "F_e2v2_test.pkl"
)

D2_train = pd.read_pickle(FEATURE_DIR / "D2_train.pkl")
D2_test = pd.read_pickle(FEATURE_DIR / "D2_test.pkl")

assert len(F_e2v2_train) == len(train) == len(D2_train)
assert len(F_e2v2_test) == len(test) == len(D2_test)
assert F_e2v2_train.columns.equals(F_e2v2_test.columns)

print("E2-v2 training features:", F_e2v2_train.shape)
print("E2-v2 test features:", F_e2v2_test.shape)

E2-v2 training features: (18000, 89)
E2-v2 test features: (10000, 89)


In [3]:
def on_off_coherence_features(D, meta):
    D = D.reset_index(drop=True)
    meta = meta.reset_index(drop=True)

    on = [0, 2, 4]
    off = [1, 3, 5]
    eps = 1e-6

    result = pd.DataFrame(index=np.arange(len(D)))

    # 1. Aggregate ON and OFF track strengths.
    on_scores = D[[f"v2_drift_p{p}_score" for p in on]].to_numpy()
    off_scores = D[[f"v2_drift_p{p}_score" for p in off]].to_numpy()

    result["on_score_mean"] = on_scores.mean(axis=1)
    result["on_score_max"] = on_scores.max(axis=1)
    result["off_score_mean"] = off_scores.mean(axis=1)
    result["off_score_max"] = off_scores.max(axis=1)

    result["on_off_score_diff"] = (
        result["on_score_mean"] - result["off_score_mean"]
    )
    result["on_off_score_ratio"] = (
        result["on_score_mean"] + eps
    ) / (result["off_score_mean"] + eps)

    # 2. Compare corresponding ON/OFF panels.
    for on_p, off_p in zip(on, off):
        on_s = D[f"v2_drift_p{on_p}_score"]
        off_s = D[f"v2_drift_p{off_p}_score"]

        result[f"pair_{on_p}_{off_p}_score_diff"] = on_s - off_s
        result[f"pair_{on_p}_{off_p}_score_ratio"] = (
            (on_s + eps) / (off_s + eps)
        )

    # 3. Agreement between the independently detected ON-panel slopes.
    on_slopes = D[
        [f"v2_drift_p{p}_slope" for p in on]
    ].to_numpy()

    result["on_slope_std"] = on_slopes.std(axis=1)
    result["on_slope_range"] = (
        on_slopes.max(axis=1) - on_slopes.min(axis=1)
    )

    # 4. Compare detected starting frequencies at approximate global times.
    on_starts = D[
        [f"v2_drift_p{p}_start" for p in on]
    ].to_numpy()

    gaps = meta["slew_gap_rows"].to_numpy(dtype=float)

    # Approximate start time for each panel, in row units.
    # This follows the challenge's proposed panel-time convention.
    global_times = np.column_stack([
        p * (16.0 + gaps) for p in on
    ])

    # If a common track continues across ON panels, its start frequencies
    # should approximately follow one line in global time.
    t_centered = global_times - global_times.mean(axis=1, keepdims=True)
    f_centered = on_starts - on_starts.mean(axis=1, keepdims=True)

    denominator = (t_centered ** 2).sum(axis=1)
    fitted_slope = (
        (t_centered * f_centered).sum(axis=1)
        / np.maximum(denominator, eps)
    )

    fitted_start = (
        on_starts.mean(axis=1)
        - fitted_slope * global_times.mean(axis=1)
    )

    predicted_starts = (
        fitted_start[:, None]
        + fitted_slope[:, None] * global_times
    )

    residuals = on_starts - predicted_starts

    result["on_global_fit_slope"] = fitted_slope
    result["on_global_fit_residual_std"] = residuals.std(axis=1)
    result["on_global_fit_residual_max"] = (
        np.abs(residuals).max(axis=1)
    )

    return result.replace([np.inf, -np.inf], np.nan).fillna(0)

In [4]:
C3_train = on_off_coherence_features(D2_train, train)
C3_test = on_off_coherence_features(D2_test, test)

print("E3 coherence features:", C3_train.shape)
print("E3 test features:", C3_test.shape)

F_e3_train = pd.concat(
    [F_e2v2_train.reset_index(drop=True), C3_train],
    axis=1
)

F_e3_test = pd.concat(
    [F_e2v2_test.reset_index(drop=True), C3_test],
    axis=1
)

assert F_e3_train.columns.is_unique
assert F_e3_train.columns.equals(F_e3_test.columns)

F_e3_train.to_pickle(FEATURE_DIR / "F_e3_train.pkl")
F_e3_test.to_pickle(FEATURE_DIR / "F_e3_test.pkl")

print("Final E3 training matrix:", F_e3_train.shape)
print("Final E3 test matrix:", F_e3_test.shape)

E3 coherence features: (18000, 17)
E3 test features: (10000, 17)
Final E3 training matrix: (18000, 106)
Final E3 test matrix: (10000, 106)


In [12]:
X_tr, X_va, y_tr, y_va = train_test_split(
    F_e3_train,
    train["label"],
    test_size=0.2,
    stratify=train["label"],
    random_state=42
)

model_e3 = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    class_weight="balanced",
    random_state=0
)

model_e3.fit(X_tr, y_tr)
pred_e3 = model_e3.predict(X_va)

score_e3 = f1_score(y_va, pred_e3, average="macro")

print("E3 validation macro-F1:", score_e3)
print(classification_report(y_va, pred_e3, zero_division=0))

E3 validation macro-F1: 0.6140945334415213
                    precision    recall  f1-score   support

     eti_candidate       0.47      0.19      0.27        96
               frb       0.69      0.71      0.70       209
             noise       0.73      0.91      0.81       842
            pulsar       0.93      0.77      0.84       326
       radar_chirp       0.75      0.73      0.74       370
  rfi_intermittent       0.73      0.52      0.61       532
    rfi_persistent       0.67      0.83      0.74       578
    satellite_pass       0.63      0.52      0.57       361
scintillating_star       0.26      0.25      0.26       286

          accuracy                           0.68      3600
         macro avg       0.65      0.60      0.61      3600
      weighted avg       0.68      0.68      0.67      3600



In [13]:
F_e3_train.to_pickle(FEATURE_DIR / "F_e3_train.pkl")
F_e3_test.to_pickle(FEATURE_DIR / "F_e3_test.pkl")

best_e3_params = {
    "max_iter": 300,
    "learning_rate": 0.05,
    "class_weight": "balanced",
    "random_state": 0,
}